In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/nikolozdodashvili/houseprice-ridge-rfe/scikitlearn/default/1/model.pkl
/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [7]:
import pickle
import numpy as np
import pandas as pd

model_path = '/kaggle/input/models/nikolozdodashvili/houseprice-ridge-rfe/scikitlearn/default/1/model.pkl'

try:
    with open(model_path, 'rb') as f:
        model = pickle.load(f)

    test_data = pd.DataFrame(np.zeros((1, 80)))
    
    prediction = model.predict(test_data)
    final_price = np.expm1(prediction)[0]
    
    print(f" ${final_price:,.2f}")

except FileNotFoundError:
    print("შეცდომა")
except Exception as e:
    print(f"შეცდომა: {e}")

 $167,820.86


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


In [8]:
import pandas as pd
test_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')
features_80 = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'BsmtFinSF1', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea', 'BsmtFullBath', 'HalfBath', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'ScreenPorch', 'PoolArea', 'TotalSF', 'TotalBathrooms', 'HouseAge', 'MSZoning_C (all)', 'MSZoning_RM', 'Alley_Pave', 'LotConfig_CulDSac', 'Neighborhood_ClearCr', 'Neighborhood_Crawfor', 'Neighborhood_Edwards', 'Neighborhood_MeadowV', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt', 'Neighborhood_Somerst', 'Neighborhood_StoneBr', 'Neighborhood_Veenker', 'Condition1_Artery', 'Condition1_Feedr', 'Condition1_RRAe', 'Condition2_PosA', 'Condition2_PosN', 'BldgType_1Fam', 'BldgType_2fmCon', 'BldgType_Twnhs', 'RoofMatl_ClyTile', 'Exterior1st_BrkComm', 'Exterior1st_BrkFace', 'Exterior1st_MetalSd', 'Exterior1st_Wd Sdng', 'Exterior2nd_CmentBd', 'Exterior2nd_Wd Sdng', 'ExterCond_TA', 'Foundation_PConc', 'BsmtQual_Ex', 'BsmtQual_TA', 'BsmtCond_Fa', 'BsmtExposure_Gd', 'BsmtFinType1_GLQ', 'Heating_GasW', 'Heating_Grav', 'HeatingQC_Ex', 'CentralAir_N', 'CentralAir_Y', 'KitchenQual_Ex', 'Functional_Maj2', 'Functional_Mod', 'Functional_Sev', 'Functional_Typ', 'GarageType_2Types', 'GarageQual_Ex', 'GarageQual_Fa', 'GarageCond_Ex', 'GarageCond_Po', 'MiscFeature_TenC', 'SaleType_ConLD', 'SaleType_New', 'SaleCondition_Abnorml', 'SaleCondition_Alloca']
test_processed = pd.get_dummies(test_df)

test_processed['TotalSF'] = test_processed['GrLivArea'] + test_processed['TotalBsmtSF']
test_processed['TotalBathrooms'] = test_processed['FullBath'] + (0.5 * test_processed['HalfBath']) + test_processed['BsmtFullBath'] + (0.5 * test_processed['BsmtHalfBath'])
test_processed['HouseAge'] = test_processed['YrSold'] - test_processed['YearBuilt']

for col in features_80:
    if col not in test_processed.columns:
        test_processed[col] = 0

X_test = test_processed[features_80].fillna(test_processed[features_80].median())


from sklearn.preprocessing import StandardScaler


scaler_temp = StandardScaler()
X_test_scaled = scaler_temp.fit_transform(X_test)

log_preds = model.predict(X_test_scaled)

print(f"Max log prediction: {log_preds.max()}")
print(f"Min log prediction: {log_preds.min()}")

log_preds = np.clip(log_preds, a_min=10, a_max=15)
final_preds = np.expm1(log_preds)

submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': final_preds
})
submission.to_csv('submission.csv', index=False)

Max log prediction: 14.178721757603139
Min log prediction: 10.930708790519532


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
